# Pandas series intro

## general tips
- With pandas boolean logic, wrap each condition in parentheses;
- datatype: `str`, `int`, `float`, `datetime[us]`, `category`
    - **string** is NOT the string type in Pandas 3.0
- Pd.series>10: returns series but `df.iloc[:,0]` a series vs `df.iloc[:,[0]]` a dataframe.
- `apply()`, or aggregation functions `sum()`, `axis=1` means apply this function along the `axis=1`, which is summing values for each row.
- `df[['sales]]` or `df.loc[:,['sales']]`

## data structures

- categorical data exclude NaN 
- date time displayed and storage format can be different
- 2 ways to set datetime and categorical types
    - directly through `dtype=datetime64[ns]`
    -  `.to_datetime()`, or `astype('category')`

In [1]:
from IPython.display import display
import pandas as pd
import numpy as np

# what is the number of category
states = ['CA', np.nan, 'NY', 'TX']
s=pd.Series(states).astype('category')
display(s.cat.categories)
len(s.cat.categories)

Index(['CA', 'NY', 'TX'], dtype='str')

3

In [2]:
# how to convert strings to datetime
string_dates_missing = ['2020-01-01 4:30', None, '2020-01-03']

pd.to_datetime(pd.Series([i[0:10] for i in string_dates_missing if i is not None]))

0   2020-01-01
1   2020-01-03
dtype: datetime64[us]

## intro to series

- series have indices
- filter series based on values or indices
- manipulation of series aligning indices
- categorical types make filtering easier
    - `s_cat=pd.CategoricalDtype(categories=[..], ordered=True)` (ascending order) and assign categories `s.astype(s_cat)`

| Function | Description |
|---|---|
| pd.Series(data=None, index=None, dtype=None, name=None, copy=False) | Create a Series from data (sequence, dict, or scalar). |
| s.index | Access index of the Series. |
| s.astype(dtype) | Cast Series to dtype. Use errors='ignore' to return the original on failure. |
| s[boolean_array] | Select values where boolean_array is True. |
| s.cat.categories | Return categories of a categorical Series. |

In [3]:
t_shirt=pd.Series([2, np.nan, 10, 5, 2], index=['brown', 'brown', 'blue', 'yellow', 'blue'])
display(t_shirt.index)
display(t_shirt.astype('category').cat.categories)

Index(['brown', 'brown', 'blue', 'yellow', 'blue'], dtype='str')

Index([2.0, 5.0, 10.0], dtype='float64')

In [4]:
# How many sales of blue and brown shirts?
display(t_shirt[['blue','brown']].sum())

# How many unique sale amounts?
display(t_shirt.nunique(dropna=True))

# How many sale amounts greater than 2?
display(t_shirt[t_shirt>2].sum())
 
# production costs from H to L are yellow, brown, and blue.
# How many shirts are less expensive to produce than yellow?
color_cat=pd.CategoricalDtype(categories=['yellow','brown', 'blue'], ordered=True)
expense=pd.Series(['brown', 'brown', 'blue', 'yellow', 'blue'], dtype='category')
display(sum(expense.astype(color_cat)>'brown'))



np.float64(14.0)

3

np.float64(15.0)

2

## intermediate series

### dunker methods

- index of the two series is aligned before performing math operations.
- `add()` vs `+`
    - `add(fill_value=0)` but `+` not allowed

| Method / Function | Operator | Description |
|---|---:|---|
| s.add(s2) | s + s2 | Adds series |
| s.sub(s2) | s - s2 | Subtracts series |
| s.mul(s2) / s.multiply(s2) | s * s2 | Multiplies series |
| s.div(s2) / s.truediv(s2) | s / s2 | Divides series |


In [5]:
s1 = pd.Series([10, 20, 30], index=['cn','us','us'])
s2 = pd.Series([    35, 44, 53], index=['us','us','jp'], name='s2')

# what is the total sales in the world?
# Note: s1+s2 returns NaN for cn and jp wrong results
s1+s2 

cn     NaN
jp     NaN
us    55.0
us    64.0
us    65.0
us    74.0
dtype: float64

In [6]:
s1.add(s2,fill_value=0)

cn    10.0
jp    53.0
us    55.0
us    64.0
us    65.0
us    74.0
dtype: float64

### aggregation functions for series

- directly apply for series
- used as keywords in `agg(['agg func word'])`

| Method / Property | Description |
|---|---|
| s.autocorr(lag=1) | Pearson correlation between s and shifted s (lag). |
| s.corr(other, method='pearson') | Correlation with other series ('pearson','spearman','kendall', or callable). |
| s.cov(other, min_periods=None) | Covariance with other series. |
| s.max(axis=None, skipna=None, level=None, numeric_only=None) | Maximum value. (min, sum, mean, median) |
| s.quantile(q=0.5, interpolation='linear') | Quantile(s); returns Series if q is list. |
| s.sem(axis=None, skipna=None, level=None, ddof=1, numeric_only=None) | Unbiased standard error of the mean. |
| s.var(axis=None, skipna=None, level=None, ddof=1, numeric_only=None) | Unbiased variance. (also std) |
| s.nunique(dropna=True) | Count of unique items. |
| s.count(level=None) | Count of non-missing items. |
| s.size | Number of items in series (property). |
| s.all()| All is truthy|
| s.any()| any is truthy|

In [7]:
# what is the mean, min and std. dev and rename as average and minimum?
# results is a series with label index
out = s1.agg(['mean', 'std', 'min'])[['mean', 'min']]
out = out.rename(index={'mean': 'average', 'min': 'minimum'})
display(out)

average    20.0
minimum    10.0
dtype: float64

In [8]:
# directly apply agg. func
pd.Series([s1.mean(), s1.std()],index=['average','stadnard deviaton'])

average              20.0
stadnard deviaton    10.0
dtype: float64

### converting data type

2 commonly used:
- `.to_datetime()`
- `.astype('str')` ('float','category')
    - `df_cat=pd.CategoricalDtype(categories=df0, sorted=True)`
    - `df['cntry']=df['cntry].astype(df_cat); df['cntry].cat.ordered`


Pandas 3.0: empty string vs missing value

- `''` is an empty string, so it is **not** missing.
- `None`, `np.nan`, and `pd.NA` are treated as missing by `isna()` and `fillna()`.
- In Pandas 3.0, inferred string data commonly displays missing values as `NaN` in the default `str` dtype.
- If your source data uses blank strings to mean missing, convert them first: `s.replace('', pd.NA)`.


In [29]:
# In pandas 3.0, None in an inferred string Series is treated as missing.
s0 = pd.Series(["a", None, "c"])
s0

0      a
1    NaN
2      c
dtype: str

In [30]:
s0.isna()

0    False
1     True
2    False
dtype: bool

In [33]:
# Summary comparison for Pandas 3.0
examples = {
    "empty string ' '": pd.Series(["A", "", "C"]),
    "None": pd.Series(["A", None, "C"]),
    "np.nan": pd.Series(["A", np.nan, "C"]),
    "pd.NA": pd.Series(["A", pd.NA, "C"]),
}

for name, s in examples.items():
    print(f"\n{name}")
    display(pd.DataFrame({
        "value": s,
        "isna": s.isna(),
        "fillna_B": s.fillna("B"),
    }))


empty string ' '


,value,isna,fillna_B
0,A,False,A
1,,False,
2,C,False,C



None


,value,isna,fillna_B
0,A,False,A
1,NaN,True,B
2,C,False,C



np.nan


,value,isna,fillna_B
0,A,False,A
1,NaN,True,B
2,C,False,C



pd.NA


,value,isna,fillna_B
0,A,False,A
1,NaN,True,B
2,C,False,C


In [ ]:
# If blank strings should behave like missing values, convert them first.
s1 = pd.Series(["A", "", None])
s1_missing = s1.replace("", pd.NA)

display(pd.DataFrame({
    "original": s1,
    "after_replace": s1_missing,
    "filled": s1_missing.fillna("B"),
}))

# to check empty strings using ==''
display(s1=='') # s1.eq('')

pd.Series(np.where(s1.eq(""), "D", s1), index=s1.index, name=s1.name)


,original,after_replace,filled
0,A,A,A
1,,NaN,B
2,NaN,NaN,B


0    False
1     True
2    False
dtype: bool

0      A
1      D
2    NaN
dtype: str

### missing value & interpolation
- `s.isna()`
- `s.fillna(0)`
- `s.dropna()`
- `s.interpolate(method='linear')`

### dedup
- `.drop_duplicates(keep='first)`
- `s[s.duplicated(keep=False)]`
- `.value_counts(dropna=False)` count all values appearing more than once

### replace
- `s.replace('',pd.NA)`

### sort idx or val
- `s.sort_index(axis=0, ascending=True)`
- `s.sort_values(axis=0, ascending=True)`

In [9]:
# duplicates (10,12), outlier (1000), missing (np.nan)
vals = pd.Series([10, 12, np.nan, 11, 10, 1000, 12, 13, 10, np.nan])
upper_bound=vals.quantile(0.8)
vals[vals>upper_bound]=999
display(vals)

0     10.0
1     12.0
2      NaN
3     11.0
4     10.0
5    999.0
6     12.0
7    999.0
8     10.0
9      NaN
dtype: float64

In [19]:
vals.info()

<class 'pandas.Series'>
RangeIndex: 10 entries, 0 to 9
Series name: None
Non-Null Count  Dtype  
--------------  -----  
8 non-null      float64
dtypes: float64(1)
memory usage: 212.0 bytes


In [10]:
display(vals.drop_duplicates(keep='first'))

0     10.0
1     12.0
2      NaN
3     11.0
5    999.0
dtype: float64

In [11]:
# determine which one has duplcates
vals[vals.duplicated(keep=False)].unique()

array([ 10.,  12.,  nan, 999.])

In [12]:
# returns a series
scount=vals.value_counts(dropna=False)
display(scount[lambda i: i>1])

10.0     3
12.0     2
NaN      2
999.0    2
Name: count, dtype: int64

In [13]:
# alternatively
scount[scount>1]

10.0     3
12.0     2
NaN      2
999.0    2
Name: count, dtype: int64

### ranking

- `s.rank(axis=0, method='dense')`
    - method handles ties: 'average','min','max','first','dense'
- sQL vs Pandas
    - `DENSE_RANK()` <-> `rank(method="dense")`
    - `RANK()` <-> `rank(method="min")` (Tied values all get the smallest rank in that tie group.)
    - `ROW_NUMBER()` <-> `rank(method="first")` after sorting appropriately (Ties are broken by the order the rows appear.)

In [147]:
from IPython.display import display
import pandas as pd
import numpy as np

df0 = pd.DataFrame({
    "firm":    ["A", "A", "A", "A", "B", "B", "B", "B", "C", "C", "C", "C"],
    "year":    [2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023],
    "region":  ["East", "East", "East", "East", "West", "West", "West", "West", "East", "East", "East", "East"],
    "sales":   [120, 135, 135, 160, 90, 110, 105, 140, 120, np.nan, 130, 130],
    "profit":  [15, 18, 18, 25, 10, 12, 11, 20, 15, 16, 19, 19]
})

In [149]:
# Within each year, rank firms by sales 
# in descending order using method="dense", 
# but only for firms in the "East" region. 
# Then create a new column that shows each East firm's rank change compared 
# with its previous year. If a prior year rank does not exist, leave it as missing.
df=df0.copy().sort_values(by=['region','firm','year'])

mask = df["region"] == "East"

df.loc[mask,'current_sales_rank']=df.loc[mask]\
    .groupby('year')['sales']\
    .rank(method='dense',ascending=False)

df.loc[mask,'previous_sales_rank']=(df.loc[mask]
    .groupby('firm')['current_sales_rank']
    .shift(1)
)

df['rank_change']=df.current_sales_rank.sub(df.previous_sales_rank)

df

,firm,year,region,sales,profit,current_sales_rank,previous_sales_rank,rank_change
0,A,2020,East,120.0,15,1.0,NaN,NaN
1,A,2021,East,135.0,18,1.0,1.0,0.0
2,A,2022,East,135.0,18,1.0,1.0,0.0
3,A,2023,East,160.0,25,1.0,1.0,0.0
8,C,2020,East,120.0,15,1.0,NaN,NaN
9,C,2021,East,NaN,16,NaN,1.0,NaN
10,C,2022,East,130.0,19,2.0,NaN,NaN
11,C,2023,East,130.0,19,2.0,2.0,0.0
4,B,2020,West,90.0,10,NaN,NaN,NaN
5,B,2021,West,110.0,12,NaN,NaN,NaN


In [69]:
# For each firm, rank profit across all years in ascending order using method="first". 
# Then, for rows where sales is not missing, create a flag indicating whether that row is simultaneously:
#  - the firm's best profit rank so far up to that year, and
#  - in the top 2 sales values for that firm across all years 
#       using method="min" with descending order.

df1=df0.copy().sort_values(by=['year','firm'])

df1['rank_profit']=df1.groupby('firm')['profit'].rank(method='first',ascending=True)

df1['best_profit_rank']=df1.groupby('firm')['rank_profit'].cummin()

df1['top_2_rank']=df1.groupby('firm')['sales'].rank(method='min', ascending=False)<=2

df1['flag']=(
    df1['sales'].notna() &
    (df1['rank_profit'] == df1['best_profit_rank']) &
    df1['top_2_rank']
)

df1.drop(columns=['best_profit_rank','top_2_rank'], inplace=True)

df1

,firm,year,region,sales,profit,rank_profit,flag
0,A,2020,East,120.0,15,1.0,False
4,B,2020,West,90.0,10,1.0,False
8,C,2020,East,120.0,15,1.0,False
1,A,2021,East,135.0,18,2.0,False
5,B,2021,West,110.0,12,3.0,False
9,C,2021,East,NaN,16,2.0,False
2,A,2022,East,135.0,18,3.0,False
6,B,2022,West,105.0,11,2.0,False
10,C,2022,East,130.0,19,3.0,False
3,A,2023,East,160.0,25,4.0,False


In [70]:
df.info()

<class 'pandas.DataFrame'>
Index: 12 entries, 0 to 7
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   firm                 12 non-null     str    
 1   year                 12 non-null     int64  
 2   region               12 non-null     str    
 3   sales                11 non-null     float64
 4   profit               12 non-null     int64  
 5   current_sales_rank   7 non-null      float64
 6   previous_sales_rank  5 non-null      float64
 7   rank_change          4 non-null      float64
dtypes: float64(4), int64(2), str(2)
memory usage: 928.0 bytes


### cat. and str.

- `str.contains('text',case=False, na=False)`
- `str.split('')[0]`
- substring: `str.slice(start=, stop=<excl. stop>)`
- concat: `df['str1']+df['str2']` or `df['str1'].str.cat(df['str2'], sep='')`

### indexing

- `.rename(index=dict_old_new)` (also rename columns dict_old_new with columns=)
- `.loc[]`
    - `series.mul(0.9).loc[lambda s_: s_>s_.mean()]`
- set numerica index
    - but index becomes a var called 'index': `reset_index()`
    - no index label to be a new column:`reset_index(drop=True)`

In [71]:
df.rename(index={0:'first'}).head(2)

,firm,year,region,sales,profit,current_sales_rank,previous_sales_rank,rank_change
first,A,2020,East,120.0,15,1.0,NaN,NaN
1,A,2021,East,135.0,18,1.0,1.0,0.0


#### iloc[] vs loc[] for series

- `[label1/position1, label2/position2]` array for selective access
- `label1:label2` or `position1:position2` for consecutive selection 
    - `'var 1' : 'var 3'` include `'var 3'` in `.loc[]` but exclude `'var 3'` in `.iloc[]`
- need to use `[:,<column slection>]` to select all rows
- `df[['col 1']]` or `df[['col 1', 'col 2']]` for columns only returning a dataframe
- `[:,[single_label/single/position]]` for dataframe
    - filtering identical to `query('SQL syntax')`

- multilevel indexing filtering (`query` and `df.var_name` short cuts not working)
    - `df.loc[:, ('CA', 'LA')]`
    - `df.loc[:, 'CA']`
    - `df.loc[:, [('CA', 'LA'), ('NY', 'NYC')]]`
    - `df.loc[:, ('CA', slice(None))]`

In [85]:
tens=pd.Series([15]*df.shape[0])
profit=df['profit']
profit

0     15
1     18
2     18
3     25
8     15
9     16
10    19
11    19
4     10
5     12
6     11
7     20
Name: profit, dtype: int64

In [86]:
profit>15 # is a pd series
try:
    profit>tens
except Exception as e:
    print(f'{e}\nb/c df is sorted \n----\n{profit.index}')

Can only compare identically-labeled Series objects
b/c df is sorted 
----
Index([0, 1, 2, 3, 8, 9, 10, 11, 4, 5, 6, 7], dtype='int64')


In [87]:
# to numpy
display(profit.to_numpy() > tens.to_numpy())

array([False,  True,  True,  True, False,  True,  True,  True, False,
       False, False,  True])

In [89]:
tens, profit=tens.align(profit)
display(profit.index)
display(tens.index)
display(profit>tens)

RangeIndex(start=0, stop=12, step=1)

RangeIndex(start=0, stop=12, step=1)

0     False
1      True
2      True
3      True
4     False
5     False
6     False
7      True
8     False
9      True
10     True
11     True
dtype: bool

In [101]:
df2=df0.rename(index={i: f'id-{i}' for i in range(12) })
df2

,firm,year,region,sales,profit
id-0,A,2020,East,120.0,15
id-1,A,2021,East,135.0,18
id-2,A,2022,East,135.0,18
id-3,A,2023,East,160.0,25
id-4,B,2020,West,90.0,10
id-5,B,2021,West,110.0,12
id-6,B,2022,West,105.0,11
id-7,B,2023,West,140.0,20
id-8,C,2020,East,120.0,15
id-9,C,2021,East,NaN,16


In [ ]:
df2.loc[['id-7','id-10']]

,firm,year,region,sales,profit
id-7,B,2023,West,140.0,20
id-10,C,2022,East,130.0,19


In [122]:
df2.iloc[[7, 10]]

,firm,year,region,sales,profit
id-7,B,2023,West,140.0,20
id-10,C,2022,East,130.0,19


In [121]:
df2.loc[['id-7','id-10'], 'firm':'sales']

,firm,year,region,sales
id-7,B,2023,West,140.0
id-10,C,2022,East,130.0


In [124]:
df2.iloc[[7,10], 0:4]

,firm,year,region,sales
id-7,B,2023,West,140.0
id-10,C,2022,East,130.0


In [120]:
df2.loc['id-7':'id-10']

,firm,year,region,sales,profit
id-7,B,2023,West,140.0,20
id-8,C,2020,East,120.0,15
id-9,C,2021,East,NaN,16
id-10,C,2022,East,130.0,19


In [126]:
df2.iloc[7:11]

,firm,year,region,sales,profit
id-7,B,2023,West,140.0,20
id-8,C,2020,East,120.0,15
id-9,C,2021,East,NaN,16
id-10,C,2022,East,130.0,19


In [119]:
df2.loc[:, 'firm':'sales']

,firm,year,region,sales
id-0,A,2020,East,120.0
id-1,A,2021,East,135.0
id-2,A,2022,East,135.0
id-3,A,2023,East,160.0
id-4,B,2020,West,90.0
id-5,B,2021,West,110.0
id-6,B,2022,West,105.0
id-7,B,2023,West,140.0
id-8,C,2020,East,120.0
id-9,C,2021,East,NaN


In [102]:
mask=(profit>15)
mask

0     False
1      True
2      True
3      True
4     False
5     False
6     False
7      True
8     False
9      True
10     True
11     True
Name: profit, dtype: bool

In [ ]:
try:
    display(df2.iloc[mask].tail(3))
except Exception as e:
    print(e)


Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).


In [109]:
display(df2.iloc[mask.to_numpy()].tail(3))
display(df2.loc[mask.to_numpy()].tail(3))
display(df2.reset_index(drop=True).loc[mask].tail(3))

,firm,year,region,sales,profit
id-9,C,2021,East,NaN,16
id-10,C,2022,East,130.0,19
id-11,C,2023,East,130.0,19


,firm,year,region,sales,profit
id-9,C,2021,East,NaN,16
id-10,C,2022,East,130.0,19
id-11,C,2023,East,130.0,19


,firm,year,region,sales,profit
9,C,2021,East,NaN,16
10,C,2022,East,130.0,19
11,C,2023,East,130.0,19


In [142]:
# same index alignment applies for assigning values
df2['flag']=mask
df2['fla2']=mask.to_numpy()
df2

,firm,year,region,sales,profit,flag,fla2
id-0,A,2020,East,120.0,15,NaN,False
id-1,A,2021,East,135.0,18,NaN,True
id-2,A,2022,East,135.0,18,NaN,True
id-3,A,2023,East,160.0,25,NaN,True
id-4,B,2020,West,90.0,10,NaN,False
id-5,B,2021,West,110.0,12,NaN,False
id-6,B,2022,West,105.0,11,NaN,False
id-7,B,2023,West,140.0,20,NaN,True
id-8,C,2020,East,120.0,15,NaN,False
id-9,C,2021,East,NaN,16,NaN,True


# Pandas DataFrame

- case when template
```python
condlist=[
    (df["cntry"].eq("CA") & df["x"].le(10)),# case 1
    (df["cntry"].eq("SA") & df["x"].le(20)),# case 2
    # add more...
    ]

choicelist=[
    2 * df["x"],   # result for case 1
    3 * df["x"],   # result for case 2
        # corresponding results...
    ]

df["y"]=np.select(condlist, choicelist,default=df["x"])
```

## merge

- not shared by both, left_only, or right_only: `df.merge(df2, how='outer', indicator=True).query('_merge != "both"')`
- types(sql keyword): `how-'inner'`(INNER JOIN), `left`(LEFT JOIN), `outer`(FULL JOIN)
- `on='col name',left_index=True,right_index=True` means using both 'col name' and index as keys but index alignment rules are not applied.

## concat

- `pd.concat([df1,df2])`(UNION ALL in SQL) combines rows while retaining indices.
    - `pd.concat([df1,df2]).drop_duplicates()` keeping one copy of the redundant (UNION in SQL) 
- intersection (INTERSECT in SQL)
    - `tmp = pd.concat([df1, df2], ignore_index=True)`
    - `tmp[tmp.duplicated(keep=False)].drop_duplicates()`
- adding columns
    - `pd.concat([df1, df2], axis=1)` adding columns
    - `df.assign(new_col_name=df2['col'])`
- replace an existing column with a transformed version
    - `df.assign(existing_one=lambda df_: df_.str.split("-")[0])`

```python
# renamed the “Political Party” column to “Party”, and then converted it to a categorical column type
df
 .iloc[1:-1]
 .rename(columns={'Political party': 'Party'})
 .assign(Party=lambda df_:df_.Party.str.replace(r'\[.*\]', '').astype('category'))
```

 ## drop

 - `.drop(index=, columns=)` to delete columns or rows   
    - use index label (use `df.index[0,2]` to access labels for known positions)


In [146]:
df2.drop(index=["id-2", "id-5"], columns=['flag','fla2'])

,firm,year,region,sales,profit
id-0,A,2020,East,120.0,15
id-1,A,2021,East,135.0,18
id-3,A,2023,East,160.0,25
id-4,B,2020,West,90.0,10
id-6,B,2022,West,105.0,11
id-7,B,2023,West,140.0,20
id-8,C,2020,East,120.0,15
id-9,C,2021,East,NaN,16
id-10,C,2022,East,130.0,19
id-11,C,2023,East,130.0,19


## sort

- `sort_values(by=['var1','var2'], ascending=False)`
- `sort_index()`